# Esercizio 2 — Sentiment Analysis con Transformer (demo)

Dimostrazione del sistema di classificazione di frasi in **positive / negative** basato su un transformer **encoder-only** (DistilBERT, famiglia BERT), fine-tunato su **SST-2**.

Il codice riutilizza i moduli in `../src`. Il modello deve essere già stato addestrato con `python src/train.py` (cartella `outputs/best_model`).

In [ ]:
import sys
from pathlib import Path

# Rendi importabili i moduli del progetto in src/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from config import Config
cfg = Config()
print('Device:', cfg.device)
print('Modello base:', cfg.model_name)

## 1. Carico il modello fine-tunato

In [ ]:
from predict import load, predict

MODEL_DIR = str(ROOT / 'outputs' / 'best_model')
tokenizer, model = load(MODEL_DIR, cfg.device)
print('Modello caricato da', MODEL_DIR)

## 2. Classifico alcune frasi di esempio

Incluso un caso con negazione (`not ... bad`), simile all'esempio della slide 11 sull'attention che ribalta il sentiment.

In [ ]:
frasi = [
    'I absolutely loved this film, it was wonderful',
    'This is the worst purchase I have ever made',
    'The plot was predictable but the acting saved it',
    'It is not bad at all, actually quite enjoyable',
    'A complete waste of time and money',
]

for text, label, conf in predict(frasi, tokenizer, model, cfg.device, cfg.max_length):
    print(f'[{label:>8}  {conf:5.1%}]  {text}')

## 3. Valutazione quantitativa sul set di validazione

Classification report (accuracy, precision, recall, F1) e matrice di confusione.

In [ ]:
import numpy as np
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from data import build_datasets
from evaluate import collect_predictions

cfg.model_name = MODEL_DIR  # usa lo stesso tokenizer del modello salvato
tokenized, _ = build_datasets(cfg)
collator = DataCollatorWithPadding(tokenizer=tokenizer)
loader = DataLoader(tokenized['validation'], batch_size=cfg.batch_size, collate_fn=collator)

y_true, y_pred = collect_predictions(model, loader, cfg.device)
target_names = [cfg.id2label[0], cfg.id2label[1]]
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

In [ ]:
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=target_names)
disp.plot(cmap='Blues', colorbar=False)
plt.title('Matrice di confusione — SST-2 (validation)')
plt.show()

## 4. Prova interattiva

Modifica la lista qui sotto con frasi tue per testare il classificatore.

In [ ]:
mie_frasi = [
    'Best customer service I have ever experienced',
    'The food was cold and the staff was rude',
]

for text, label, conf in predict(mie_frasi, tokenizer, model, cfg.device, cfg.max_length):
    print(f'[{label:>8}  {conf:5.1%}]  {text}')